# B3: تقييم إجهاد المنطقة على دقة أعلى (بدون إعادة تدريب)

**السياق**: `region_cauchy_field_rel` (المقياس المحصّن من artifacts القياس) طلع mean=71.77%/median=68.45% على checkpoint_50000.pt عند **دقة الإنتاج** (6,840 عنصر، بس **6 نقاط Gauss** بمنطقة الأخدود). هاد المقياس صحّح كيف نقارن الحقلين، بس لسا بيقارنهم عند 6 نقاط بس -- ما بيجاوب سؤال: هل 6 نقاط كافية أصلاً لأخذ عينة من المنطقة؟

**هاد الدفتر**: ياخد عيّنة فرعية من نفس الـ100 عينة held-out، **بلا إعادة تدريب**: (1) بيستكمل (interpolate) حقل E/nu الموجود أصلاً لكل عينة على شبكة أنعم بكتير (41×36×19 = 43,400 عنصر، **36 نقطة Gauss** بالمنطقة بدل 6) -- نفس آلية الاستكمال (`_field_interpolator`) يلي المشروع مستخدمها أصلاً لمقارنة الدقات المختلفة (نفس العينة الفيزيائية، مش عينة عشوائية جديدة)؛ (2) بيحل FEM حقيقي جديد على الشبكة الأنعم بنفس المادة المستكملة ونفس phi؛ (3) بيستعلم نفس الشبكة المدرّبة (checkpoint_50000، بلا أي تغيير) zero-shot عند نقاط الشبكة الأنعم؛ (4) بيحسب `region_cauchy_field_rel` (نفس الدالة، بلا أي تعديل) عند منطقة الشبكة الأنعم.

**لو الرقم الجديد نزل كتير عن 71.77%** -- جزء كبير من الفجوة كان بسبب قلة نقاط العينة عند دقة الإنتاج. **لو ظل قريب من 71.77%** -- الفجوة حقيقية ومستقلة عن دقة التقييم.

**قيد معروف بالتصميم**: استكمال حقل GRF الخشن على شبكة أنعم ما بيرجع عينة GRF أدق فعليًا (مولّد GRF الطيفي بيربط الضجيج مباشرة بحجم الشبكة المستهدفة، فنفس الـseed بدقة مختلفة بيعطي حقل عشوائي مختلف تمامًا -- تأكدنا من هاد بقراءة `grf.py` مباشرة قبل ما نفترض إن الاستكمال هو الحل الصح). استكمال العينة الموجودة أصلاً هو الطريقة الصحيحة للحفاظ على نفس المسألة الفيزيائية وبس تنعيم الشبكة/التقييم -- بالضبط السؤال المطروح هون -- بس هاد يعني إنه هاد الاختبار بيفحص "هل التقييم الأنعم بيغيّر الدقة المقاسة لنفس الحقل"، مش "هل التدريب على بيانات فيزيائية أدق فعليًا رح يغيّر تنبؤات الشبكة".

**اتفحص محليًا على CPU قبل هيك بثلاث خطوات حقيقية**: (1) فحص identity -- استكمال حقل على نفس شبكته الخشنة بيرجعه بالضبط (فرق = 0.0)؛ (2) فحص الحدود -- الاستكمال على شبكة أنعم بيخلي E/nu بحدودهم الفيزيائية الأصلية؛ (3) تشغيلة كاملة بحجم toy (بيانات صغيرة -> تدريب قصير -> إعادة حل على شبكة أنعم -> استعلام zero-shot -> `region_cauchy_field_rel`) بلا أي أخطاء.

**التكلفة الحقيقية مش معروفة لسا على GPU حقيقي**: كل عينة من الـ`--n_samples` تحتاج حل FEM جديد فعليًا عند 43,400 عنصر (بدل 6,840). لو الوقت طويل جدًا، وقف وشغّل بعدد عينات أقل (حتى 3-5 عينات بتعطي إشارة حقيقية، بس أضجّ شوي).

**تحديث 2026-10-01**: هاد الدفتر هلأ كمان بيحسب دقة الإزاحة والطاقة ورد الفعل (مش بس إجهاد المنطقة) مقابل نفس المرجع الأنعم -- مطلوبة لمقارنة accuracy-matched يلي طلبها تيمون (رقم الشبكة NO مقابل أرخص FEM بنفس دقتها، مو مقابل شبكة التدريب 6,840 عنصر).


In [ ]:
# =====================================================================
#  CELL -- B3: finer-resolution region-stress evaluation, to separate
#  an evaluation-resolution artifact from a genuine network accuracy
#  gap. See evaluate_B3_qois_finer.py's own module docstring for the
#  full derivation.
#
#  Context: region_cauchy_field_rel (the artifact-free full-tensor
#  field metric) came back mean=71.77%/median=68.45% on the real
#  checkpoint at PRODUCTION resolution (6,840 elements, only 6 Gauss
#  points in the region). That metric fixed HOW the two fields are
#  compared, but still compares them at only 6 points -- it does not
#  test whether 6 points is enough to SAMPLE the region in the first
#  place. This notebook re-solves a SUBSET of the same held-out samples
#  at a much finer mesh (36 Gauss points in the region at (41,36,32),
#  vs. 6 at production resolution), interpolating each sample's
#  already-generated material field onto the finer mesh (no new random
#  draw -- the SAME physical sample, just refined), and queries the
#  SAME trained network zero-shot at that finer mesh. If the finer
#  region_cauchy_field_rel is much lower than 71.77%, the production-
#  resolution number was largely an evaluation-sampling artifact. If it
#  stays similarly high, the gap is real and resolution-independent.
#
#  Verified locally on CPU first (toy scale, real solves/real network,
#  not stubs): (1) an identity check -- interpolating a field back onto
#  its own coarse grid reproduces it exactly; (2) a bounds check --
#  interpolating onto a finer grid keeps E/nu within their original
#  physical range; (3) a full toy end-to-end run exercising every real
#  code path (tiny dataset -> brief training -> finer re-solve ->
#  zero-shot query -> region_cauchy_field_rel), no errors.
#
#  COST IS NOT YET KNOWN ON REAL GPU DATA: each of the --n_samples
#  samples below requires a genuinely NEW nonlinear FEM solve at
#  43,400 elements (vs. 6,840 at production resolution) plus one
#  network forward pass. If this is taking unexpectedly long, interrupt
#  and re-run with a smaller --n_samples (even 3-5 samples already give
#  a real, if noisier, read on whether the gap shrinks).
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
# Idempotent: mount() raises "Mountpoint must not already contain files"
# if re-run in a still-alive runtime that already mounted Drive.
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

R = '/content/drive/MyDrive/pfem_run'
CHECKPOINT = f'{R}/b3_training_normalized/checkpoint_50000.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{R}/b3_training_normalized/qois_finer_resolution.json'

assert os.path.exists(CHECKPOINT), f'checkpoint not found: {CHECKPOINT}'
assert os.path.exists(DATASET), f'clean held-out dataset not found: {DATASET}'

sys.argv = [
    'evaluate_B3_qois_finer.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
    '--fine_resolution', '41', '36', '32',
    '--n_samples', '10',
]
from omar_pfem.evaluate_B3_qois_finer import main
main()

print('\nDone.')


## break-even (resolution-matched + accuracy-matched)

بعد ما الخلية فوق تخلّص (بتنتج `qois_finer_resolution.json` بالأرقام الجديدة)، هاي الخلية بتحسب:
1. **resolution-matched**: NO وFEM عند نفس دقة التدريب (6,840 عنصر) -- جاهز فورًا، أرقام حقيقية موجودة أصلاً.
2. **accuracy-matched**: أرخص شبكة FEM بنفس دقة NO على الأقل (الاثنين مقابل نفس المرجع الأنعم 243,360 عنصر) -- بتستخدم جدول التقارب الحقيقي الموجود أصلاً (`mesh_convergence_extended.json`, 2026-09-21).

إجهاد المنطقة (region stress) ما داخل بهالمقارنة عمدًا -- تيمون قرر نبلّغه كـ limitation موثّقة، مش جزء من مقارنة break-even.


In [ ]:
# =====================================================================
#  CELL -- B3: break-even analysis (resolution-matched + accuracy-matched),
#  per Timon's "finish the paper" request (2026-10-01): complete the
#  accuracy-matched FEM/VINO comparison and break-even analysis for the
#  QoIs that are reproduced well (displacement, energy, reaction).
#
#  Resolution-matched (NO and FEM both at B3's own 6,840-element training
#  resolution) needs no new data -- it is pure arithmetic on already-
#  verified numbers. Accuracy-matched (cheapest FEM mesh at least as
#  accurate as the NO, both judged against the SAME independent
#  243,360-element fine reference) uses the qois_finer_resolution.json
#  the previous cell just produced, which now also reports displacement/
#  energy/reaction error vs. that fine reference (added 2026-10-01 to
#  evaluate_B3_qois_finer.py -- previously that script only reported
#  region stress). See break_even_B3.py's own module docstring for the
#  full derivation and the sourced convergence-ladder table it uses.
# =====================================================================
import sys

sys.argv = [
    'break_even_B3.py',
    '--fine_qois_json', OUT_JSON,
    '--out_json', f'{R}/b3_training_normalized/break_even_B3.json',
]
from omar_pfem.break_even_B3 import main
main()

print('\nDone.')
